# Notebook 15.1  Instruction routing and honest, per-dialect evaluation for an audio-language model

**Goal.** Show how one audio-language model serves many speech tasks through instructions, and build the per-dialect, provenance-aware evaluation harness Chapter 15 argues for.

**What runs here.** Everything runs with no downloads. We represent a small set of audio clips by their task labels and dialect tags (a synthetic fallback), route free-text instructions in English and Arabic to tasks, run a mock multitask model, and score each task with its proper metric, broken down per dialect. We then add evaluation-integrity checks that flag the shortcuts the chapter warns about: model-generated labels, synthesized test audio, and train/test overlap. The markdown says exactly where a real audio-LLM and dataset would plug in.

**The point.** Capability is easy to demo and hard to measure. A single average number can hide weak dialects and inflated scores; the honest report is per dialect, per task, on real speech, with the data provenance stated.

## 1. Setup

In [ ]:
import re
from collections import defaultdict, Counter
print('ready')

## 2. The speech-LLM task taxonomy and an instruction router

One model is asked, by instruction, to do many tasks. A real model reads the instruction with its language model; here we use a transparent keyword router (English and Arabic) as a baseline, exactly the baseline the chapter says is not enough. We will see where it misroutes.

In [ ]:
TASKS = ['asr', 'translation', 'speech_qa', 'dialect_id', 'emotion']

# keyword rules: task -> list of (English/Arabic) trigger substrings
RULES = {
    'asr':        ['transcribe', 'transcription', 'write down', 'اكتب', 'فرغ', 'فرّغ', 'نص'],
    'translation':['translate', 'translation', 'ترجم', 'ترجمة', 'to english', 'الى الانجليزية'],
    'dialect_id': ['dialect', 'which variety', 'لهجة', 'اللهجة'],
    'emotion':    ['emotion', 'feeling', 'sentiment', 'مشاعر', 'انفعال', 'شعور'],
    'speech_qa':  ['?', 'question', 'what', 'when', 'who', 'how many', 'هل', 'ما ', 'متى', 'كم', 'لماذا'],
}

def route(instruction):
    text = instruction.lower()
    # priority: explicit task verbs before the generic question cue
    for task in ['asr', 'translation', 'dialect_id', 'emotion', 'speech_qa']:
        if any(kw in text for kw in RULES[task]):
            return task
    return 'asr'   # default fallback

tests = [
    ('Transcribe the speech.', 'asr'),
    ('فرّغ المقطع الصوتي', 'asr'),
    ('Translate to English.', 'translation'),
    ('ترجم الكلام إلى الإنجليزية', 'translation'),
    ('Which dialect is this?', 'dialect_id'),
    ('ما هي اللهجة؟', 'dialect_id'),
    ('What emotion is expressed?', 'emotion'),
    ('متى الموعد؟', 'speech_qa'),
    ('Translate and tell me the dialect.', 'translation'),   # compound: only one task wins
    ('وش قال؟', 'speech_qa'),                                  # dialectal 'what did he say' -> ambiguous ASR/QA
]
correct = 0
for instr, gold in tests:
    pred = route(instr)
    ok = (pred == gold)
    correct += ok
    print(('OK ' if ok else 'XX '), '%-12s pred=%-11s' % (gold, pred), '|', instr)
print('\nrouter accuracy: %d/%d' % (correct, len(tests)))

The router misses two cases on purpose: a *compound* instruction ("translate and tell me the dialect") has two tasks but a keyword rule returns only one, and a *dialectal* phrasing (“وش قال؟”, wesh gaal, ‘what did he say’) is genuinely between ASR and speech QA. This is the lesson: instruction following needs the language model's understanding, not a fixed rule table.

## 3. A mock multitask model and per-dialect scoring

We now evaluate a multitask model. Each test item carries a dialect tag and a gold answer. We score ASR by word error rate, dialect identification and emotion by accuracy, and report **per dialect**, never pooled. The mock model is deliberately weaker on one dialect to show why pooling hides failure. Replace `mock_model` with a real audio-LLM and this harness is unchanged.

In [ ]:
def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    # Levenshtein on words
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1]==h[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(r)][len(h)] / max(1, len(r))

# synthetic evaluation set: (task, dialect, gold, provenance)
# provenance: 'recorded' (real speech) or 'tts' (synthesized)
eval_set = [
    ('asr',        'gulf',     'وش الاخبار اليوم',        'recorded'),
    ('asr',        'egyptian', 'الجو حلو النهارده',        'recorded'),
    ('asr',        'levantine','شو عم تعمل هلق',           'recorded'),
    ('dialect_id', 'gulf',     'gulf',                     'recorded'),
    ('dialect_id', 'egyptian', 'egyptian',                 'recorded'),
    ('dialect_id', 'levantine','levantine',                'recorded'),
    ('emotion',    'gulf',     'happy',                    'recorded'),
    ('emotion',    'egyptian', 'angry',                    'recorded'),
    ('emotion',    'levantine','neutral',                  'recorded'),
]

def mock_model(task, dialect, gold):
    # deliberately weaker on Levantine to expose per-dialect gaps
    weak = (dialect == 'levantine')
    if task == 'asr':
        if weak:
            return gold.split()[0] + ' XXX ' + ' '.join(gold.split()[2:])  # an error in the middle
        return gold
    else:
        if weak and task == 'dialect_id':
            return 'gulf'      # confuses Levantine with Gulf
        if weak and task == 'emotion':
            return 'happy'     # wrong
        return gold

# score per (task, dialect)
by = defaultdict(list)
for task, dialect, gold, prov in eval_set:
    pred = mock_model(task, dialect, gold)
    if task == 'asr':
        score = wer(gold, pred); metric='WER'
    else:
        score = 1.0 if pred == gold else 0.0; metric='acc'
    by[(task, metric)].append((dialect, score))

print('PER-DIALECT RESULTS (the honest report)')
for (task, metric), items in by.items():
    print('\n%s (%s):' % (task, metric))
    for dialect, score in items:
        print('   %-11s %s = %.2f' % (dialect, metric, score))
    pooled = sum(s for _,s in items)/len(items)
    print('   %-11s pooled %s = %.2f   <-- hides the per-dialect gap' % ('[ALL]', metric, pooled))

## 4. Evaluation-integrity checks

Honest evaluation is not only about metrics; it is about how the test data was made. These checks flag the five shortcuts from the chapter's cautionary box. They are simple, but in practice they catch most inflated Arabic speech-LLM numbers.

In [ ]:
def integrity_report(eval_set, labels_source, train_ids, test_ids):
    issues = []
    # (2) synthesized test audio
    tts = [d for *_, d in [(x[0],x[3]) for x in eval_set] if d == 'tts']
    n_tts = sum(1 for x in eval_set if x[3] == 'tts')
    if n_tts:
        issues.append('%d/%d test items use synthesized (TTS) audio, not real speech' % (n_tts, len(eval_set)))
    # (1) model-generated gold labels
    if labels_source != 'human':
        issues.append('gold labels were produced by %r, not human annotators' % labels_source)
    # (4) train/test overlap (ASR contamination)
    overlap = set(train_ids) & set(test_ids)
    if overlap:
        issues.append('train/test overlap on ids: %s' % sorted(overlap))
    return issues

# a clean run vs a contaminated run
clean = integrity_report(eval_set, labels_source='human', train_ids={1,2,3}, test_ids={4,5,6})
bad_set = eval_set[:6] + [('asr','gulf','وش الاخبار اليوم','tts')]
bad = integrity_report(bad_set, labels_source='gpt-4', train_ids={1,2,3}, test_ids={3,4,5})

print('CLEAN evaluation issues:', clean if clean else 'none -> trustworthy')
print()
print('SUSPECT evaluation issues:')
for i in bad: print('  -', i)

## 5. What this shows, and where a real model plugs in

Three lessons. First, one model can serve many tasks, but routing by instruction is real work: a keyword baseline misroutes compound and dialectal instructions, which is why instruction following needs a language model, not a rule table. Second, a single pooled score hides the dialect that fails; the only honest report is per dialect and per task, with the metric proper to each. Third, strong-looking numbers can come from weak evaluation, so the integrity checks, real recorded audio, human labels, and no train/test overlap, are part of the result, not an afterthought.

To make this real, replace the mock pieces with a deployed audio-language model and a real dataset: route instructions with the model itself, run it over recorded, dialect-tagged Arabic speech (for grounded spoken question answering, instruction-style speech sets and EverydayMMQA are starting points, subject to their licenses), and keep the per-dialect breakdown and the provenance checks exactly as they are. The harness is the contribution; the model is swappable.